# Milestone 3: Baseline Reproduction
CSCI 3052U Machine Learning I,
Jason Badwal, Cort Auger, Saad Siddiqui, Adlene Oudahmane

For Milestone 3, we put together a working data pipeline, a naive benchmark, a baseline using methods from the course, a reproducible environment, a table comparing the baselines, an analysis of where our results don't line up, and a start/stop/continue plan. We go through each of these in order below. We put all the actual code in `src/` so that the notebook, the script, and the tests all run the exact same pipeline:


| File | What it does |
|---|---|
| `src/data.py` | The Milestone 2 cleaning, design matrix, split and normalization, turned into functions |
| `src/models.py` | Softmax regression with L2 regularization and gradient descent, written from the Week 5 slides |
| `src/evaluate.py` | Macro-F1 (our primary metric), accuracy, per-class scores, confusion matrix |
| `src/run_baselines.py` | Regenerates every number and figure in `results/` (`python -m src.run_baselines`) |
| `tests/test_pipeline.py` | Split, leakage, normalization, gradient and determinism checks |

Where each method comes from:
- [Lab 2] / [Week 2]: cleaning, one-hot encoding, the design matrix, the accuracy trap on imbalanced data
- [Lab 3] / [Week 3]: the stratified split, normalization with training-set mean/std plus `epsilon`, choosing hyperparameters on validation only, using the test set once (W3 p.51–52)
- [Lab 4]: the `build_all_models` dictionary search and the "highest validation score so far" loop
- [Week 4] / [Week 5]: L2 regularization, gradient descent and the learning rate, the training curve, and softmax regression with cross-entropy

**The test set isn't used anywhere in this milestone.** Every score reported here comes from the validation set. We're saving the test set for a single final evaluation in Milestone 5.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

# the notebook lives in notebooks/, so put the repo root on the path to import src
sys.path.insert(0, str(Path.cwd().parent))

from src.data import (CLASS_COLORS, CLASS_ORDER, DATA_PATH, DATA_SHA256, RANDOM_STATE,
                      file_sha256, load_splits)
from src.evaluate import evaluate, plot_confusion
from src.models import SoftmaxRegression
from src.run_baselines import run_naive

RESULTS = Path.cwd().parent / "results"

: 

## Part 1 - Working data pipeline
`load_splits()` runs our Milestone 2 pipeline starting from the raw CSV. It loads the data and checks its SHA-256 fingerprint, then cleans out the five kinds of bad data we covered in Week 2. Next it one-hot encodes everything into the design matrix and does a stratified 70/15/15 split with seed 42. it then runs our split-integrity checks from M2, and finally it normalizes the continuous columns using the mean and standard deviation from the **training set only**.

In [ ]:
print("Dataset SHA-256 matches M2:", file_sha256(DATA_PATH) == DATA_SHA256)

s = load_splits(seed=RANDOM_STATE)
X_train, X_valid, t_train, t_valid = s["X_train"], s["X_valid"], s["t_train"], s["t_valid"]

print("Design matrix X:", s["X"].shape)
print("train / valid / test:", X_train.shape, X_valid.shape, s["X_test"].shape)

In [ ]:
# the same class proportions in every split (stratified), as in the M2 split table
pd.DataFrame({
    "train": t_train.value_counts(normalize=True),
    "valid": t_valid.value_counts(normalize=True),
    "test":  s["t_test"].value_counts(normalize=True),
}).rename(index=dict(enumerate(CLASS_ORDER))).round(3)

**Answer:** The design matrix has 169 columns and the split is 3,096 / 664 / 664, the same as Milestone 2. `tests/test_pipeline.py` turns these numbers into asserts, together with the disjoint-split, no-2nd-semester and normalization checks.

## Part 2 - Naive benchmarks
We use two naive benchmarks:
- **Majority class** (`DummyClassifier(strategy="most_frequent")`) always predicts Graduate. This is the M2 benchmark.
- **Stratified random** (`DummyClassifier(strategy="stratified")`) guesses a label at random using the training class rates. It shows what "no information about the student" scores while still predicting every class.

Both can be worked out on paper before running anything, which gives us an expected value to compare against.

**Majority class.** Let $p_G$ be the share of Graduates. Graduate gets recall 1 and precision $p_G$, and the other two classes get F1 = 0:

$$F1_{G} = \frac{2 \cdot p_G \cdot 1}{p_G + 1}, \qquad \text{macro-F1} = \frac{1}{3}F1_G = \frac{1}{3}\cdot\frac{2(0.5)}{1.5} = 0.222$$

**Stratified random.** If the guess for each student is independent of their true class and class $c$ is guessed with probability $p_c$, then on average precision$_c$ = $p_c$ and recall$_c$ = $p_c$, so $F1_c \approx p_c$ and

$$\text{macro-F1} \approx \frac{1}{3}(p_D + p_E + p_G) = \frac{1}{3} \approx 0.333$$

In [ ]:
majority_row, stratified_row = run_naive(s, RANDOM_STATE)
for r in [majority_row, stratified_row]:
    print(f"{r['model']:18s} valid macro-F1 = {r['valid_macro_f1']:.4f}   valid accuracy = {r['valid_accuracy']:.3f}")

**Answer:** The majority benchmark scores **0.222**, which matches both the analytic value and our M2 number. It also gets 50% accuracy, which is exactly the accuracy trap from Lab 2 / Week 2: a model that completely ignores two of the three classes still looks about as good as a coin flip if you only check accuracy. The stratified benchmark scores 0.313 on this particular split. That's a bit under the 0.333 we expected, but it's only one random draw of 664 guesses. Averaged over 10 seeds it comes out to 0.343 ± 0.012 (see Part 7), which is close to 1/3.

## Part 3 - Course-level baseline: softmax regression [Week 5 p.88–96]
Our proposal named logistic regression as the baseline. We have three classes, so we use its multi-class version from Week 5, **softmax regression**. We wrote it ourselves in NumPy (`src/models.py`) following the slides, then checked it against scikit-learn (Part 6).

With the dummy feature $x_0 = 1$ absorbing the bias (W5 p.45) and one-hot targets $\mathbf{t}$ (W5 p.90):

$$\mathbf{z} = \mathbf{W}\mathbf{x}, \qquad y_k = \text{softmax}(\mathbf{z})_k = \frac{e^{z_k}}{\sum_{k'} e^{z_{k'}}}, \qquad \mathcal{L}_{CE}(\mathbf{y}, \mathbf{t}) = -\sum_k t_k \log y_k$$

The cost adds the Week 4 L2 regularizer (W4 p.75). We don't penalize the bias:

$$\mathcal{J}(\mathbf{W}) = \frac{1}{N}\sum_{i=1}^N \mathcal{L}_{CE}(\mathbf{y}^{(i)}, \mathbf{t}^{(i)}) + \frac{\lambda}{2}\sum_{j \ge 1}\sum_k w_{kj}^2$$

The gradient of the CE part for one example is $(y_k - t_k)\,\mathbf{x}$ (W5 p.96), the same form as linear regression (W5 p.82). Vectorized over the training set, the gradient descent step is

$$\mathbf{W} \leftarrow \mathbf{W} - \alpha\left[\frac{1}{N}\mathbf{X}^\top(\mathbf{Y} - \mathbf{T}) + \lambda\mathbf{W}\right]$$

The $\lambda\mathbf{W}$ term is the weight decay from W4 p.88. We start from $\mathbf{W} = 0$ (W5 p.79). The cross-entropy loss is convex in $\mathbf{W}$, so with a small enough $\alpha$, gradient descent heads to the global minimum.

Why this as the baseline: W5 p.73 calls logistic regression "a good starting point" that is "easy to implement and interpret" and gives probabilities. The same slide warns that it "requires feature scaling", which our pipeline already does, and that it assumes a linear decision boundary, which is the limitation M4's kNN and decision trees will test.

## Part 4 - Training curve and the learning rate [Week 4 p.83–90]
W4 p.90 says to grid-search the learning rate ("try 0.1, 0.03, 0.01, ...") and to diagnose optimization with a training curve.

In [ ]:
plt.figure(figsize=(6, 3.5))
for a in [0.3, 0.1, 0.03, 0.01]:
    m = SoftmaxRegression(alpha=a, lam=0.0, n_iters=5000).fit(X_train, t_train)
    plt.plot(m.cost_history, label=f"alpha = {a}")
plt.title("Training curve, lambda = 0")
plt.xlabel("Gradient descent iteration"); plt.ylabel("Training cost J"); plt.legend()
plt.show()

**Answer:** Every curve goes down at every step (our `test_cost_decreases` test checks this), so none of these learning rates is too big. Smaller values of $\alpha$ are just slower: even after 5,000 steps, $\alpha = 0.01$ is still well above $\alpha = 0.3$. This comes up again in Part 6, since a curve that's still sloping down hasn't actually converged yet.

## Part 5 - Hyperparameter tuning on the validation set [Lab 4]
`src/run_baselines.py` fits one model for every $(\alpha, \lambda)$ pair, the same way `build_all_models` did in Lab 4, and keeps the pair with the highest **validation** macro-F1. Search space: $\alpha \in \{0.3, 0.1, 0.03, 0.01\}$ and $\lambda \in \{0, 0.001, 0.01, 0.1, 1\}$, each run for 5,000 iterations. That's 20 models, about 1 minute. The full grid is saved in `results/tuning_grid.csv`, so we read it back here instead of refitting:

In [ ]:
grid = pd.read_csv(RESULTS / "tuning_grid.csv")
grid.pivot(index="lambda", columns="alpha", values="valid_macro_f1").round(3)

In [ ]:
optimal_alpha = 0
optimal_lambda = 0
highest_validation_f1_so_far = 0

for _, r in grid.iterrows():
    if r["valid_macro_f1"] > highest_validation_f1_so_far:
        highest_validation_f1_so_far = r["valid_macro_f1"]
        optimal_alpha = r["alpha"]
        optimal_lambda = r["lambda"]

print("Optimal alpha:", optimal_alpha)
print("Optimal lambda:", optimal_lambda)
print("Optimal validation macro-F1:", round(highest_validation_f1_so_far, 4))

In [ ]:
# train vs valid across lambda (the Lab 3 train/valid curve)
g = grid[grid["alpha"] == optimal_alpha]
plt.figure(figsize=(6, 3.5))
plt.plot(g["lambda"].astype(str), g["train_macro_f1"], marker="o", label="Train")
plt.plot(g["lambda"].astype(str), g["valid_macro_f1"], marker="o", label="Valid")
plt.axhline(majority_row["valid_macro_f1"], color="grey", linestyle="--", label="Majority class (valid)")
plt.xlabel("lambda"); plt.ylabel("Macro-F1"); plt.legend()
plt.show()

**Answer:** Validation macro-F1 is highest with **no** L2 penalty at all ($\lambda = 0$, $\alpha = 0.3$: 0.661), and it drops steadily as $\lambda$ increases, all the way down to 0.442 at $\lambda = 1$. The train and validation curves stay pretty close together (a gap of around 0.04), so the baseline isn't overfitting. If anything it's closer to the "high bias" end that W4 p.59 describes for linear models, so penalizing the weights just makes it underfit even more.

There's a catch with $\lambda = 0$. With no penalty, the cost has no finite minimum on this data (Part 6), so the model we pick depends on stopping after 5,000 iterations. W5 p.124 calls this **early stopping**, and it acts as a regularizer. The tuned result is therefore "5,000 GD steps at $\alpha = 0.3$, $\lambda = 0$", and that's exactly what we record.

In [ ]:
best = SoftmaxRegression(alpha=optimal_alpha, lam=optimal_lambda, n_iters=5000).fit(X_train, t_train)
y_valid = best.predict(X_valid)
scores = evaluate(t_valid, y_valid)

pd.DataFrame({c: [scores[f"precision_{c}"], scores[f"recall_{c}"], scores[f"f1_{c}"]] for c in CLASS_ORDER},
             index=["precision", "recall", "F1"]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
plot_confusion(t_valid, y_valid, "Softmax regression, validation", ax=ax)
plt.show()

**Answer:** The model does well on Graduate (F1 0.85) and Dropout (F1 0.73). **Enrolled is the weak class**, with an F1 of 0.41 and recall of 0.35. Out of the 119 Enrolled students in validation, 36 get predicted as Dropout and 41 as Graduate. We expected this back in M2: Enrolled is the smallest class (18%), and a delayed student's first semester can look a lot like either of the other two outcomes. For an advising office, 70% Dropout recall means roughly 3 in 10 future dropouts wouldn't get flagged. Handling the imbalance for Enrolled is the first thing on our Start list.

## Part 6 - Baseline table and discrepancy analysis
`python -m src.run_baselines` writes the table to `results/baseline_table.csv` / `.md`:

In [ ]:
table = pd.read_csv(RESULTS / "baseline_table.csv")
table[["model", "train_macro_f1", "valid_macro_f1", "valid_macro_f1_mean_10_seeds",
       "valid_macro_f1_std_10_seeds", "valid_accuracy", "valid_recall_Dropout",
       "valid_recall_Enrolled"]].round(3)

The rubric asks us to compare the reproduced result with the expected or published result. Our expected values come from three places: the numbers we recorded in Milestone 2, the analytic values in Part 2, and a reference implementation (scikit-learn) of the same model.

In [ ]:
with open(RESULTS / "discrepancy.json") as f:
    d = json.load(f)

pd.DataFrame([
    ["Majority valid macro-F1", d["majority_valid_macro_f1"]["m2_recorded"], round(d["majority_valid_macro_f1"]["reproduced"], 4), "analytic 0.222"],
    ["Stratified valid macro-F1", "~0.333 (analytic)", f'{d["stratified_valid_macro_f1"]["reproduced_mean_10_seeds"]:.3f} ± {d["stratified_valid_macro_f1"]["reproduced_std_10_seeds"]:.3f}', "seed 42 alone: 0.313"],
    ["Design matrix columns", d["n_features"]["m2_recorded"], d["n_features"]["reproduced"], ""],
    ["Split sizes", d["split_sizes"]["m2_recorded"], d["split_sizes"]["reproduced"], ""],
    ["Success bar (valid macro-F1)", f'>= {d["success_bar"]["m2_preregistered"]}', round(d["success_bar"]["softmax_valid_macro_f1"], 4), "met" if d["success_bar"]["met"] else "NOT met"],
], columns=["quantity", "expected", "reproduced", "note"])

In [ ]:
# NumPy GD vs sklearn LogisticRegression at the same lambda (C = 1 / (lambda * N))
pd.read_csv(RESULTS / "sklearn_check.csv").round(4)

**Answer (discrepancy analysis):**
- **Naive benchmark and pipeline: exact match.** The majority benchmark gives 0.222, the same as the M2 number and the analytic value. The 169 features and the 3,096 / 664 / 664 split are identical to M2. The analytic 0.2220 uses the training share of Graduates (0.499); on the validation share (exactly 0.500) it's 0.2222, which is what we measured.
- **Stratified random: matches on average.** One seed came out low (0.313), and the 10-seed mean (0.343 ± 0.012) is within one standard deviation of 1/3. This is just the randomness of a single draw, and it's the reason we report seeds.
- **NumPy vs scikit-learn: exact match wherever the problem has a single answer.** For every $\lambda > 0$ the two implementations agree on 99.7–100% of validation predictions, and for $\lambda \ge 0.01$ they reach the same cost and essentially the same weights. That's what convexity predicts (W5 p.79), and together with the finite-difference gradient check in `tests/` it tells us our derivation and code are right. The one real discrepancy is at $\lambda = 0$, our chosen setting: 95.3% agreement, and sklearn's final cost is lower (0.541 vs 0.556). The cause is that without a penalty the cost has no finite minimum, so neither optimizer converges. sklearn hit its 1,000-iteration cap, and a 10,000-iteration run took 200 s without stopping. Each optimizer stops at a different point on the way, so their predictions differ. Our baseline is the early-stopped GD model, and its result depends on `n_iters`.
- **Published result.** The original paper on this dataset (Martins et al., 2021) is behind Springer's paywall and we haven't been able to read it, so we don't quote any numbers from it. The 2022 data descriptor (Realinho et al., *Data* 7(11):146), which we did read, reports permutation feature importance but no model scores. The descriptor's feature-importance analysis uses all features, including the 2nd semester, and its top feature for all four models is *Curricular units 2nd sem (approved)*. Our early-warning task drops those columns (M2 3.5). To measure how much that costs us, we refit the same baseline with them added back: validation macro-F1 goes from 0.661 to 0.695. Any published score that uses all features should be read with that gap of about 0.03 in mind, and it isn't a target we can match without leaking information from after our prediction point.
- **Environment.** M2 ran on Python 3.13.15, pandas 2.2.3, numpy 2.1.3 and scikit-learn 1.6.1, and `requirements.txt` pins exactly those. The one gap we found: M2 says the split indices were saved, but they were written under the git-ignored `data/` folder and never committed. `results/split_indices.json` is now committed.

## Part 7 - Variability over seeds
One split can be lucky or unlucky (M2 risk table). We rebuilt the split with seeds 0–9 and refit the chosen configuration on each, without re-tuning:

In [ ]:
seeds = pd.read_csv(RESULTS / "seed_runs.csv")
seeds.groupby("model")[["valid_macro_f1", "valid_recall_Dropout", "valid_recall_Enrolled"]].agg(["mean", "std"]).round(3)

In [ ]:
plt.figure(figsize=(6, 3.5))
for i, (name, g) in enumerate(seeds.groupby("model")):
    plt.scatter(g["valid_macro_f1"], [i] * len(g), label=name)
plt.yticks(range(3), sorted(seeds["model"].unique()))
plt.axvline(0.272, color="grey", linestyle="--", label="M2 success bar (0.272)")
plt.xlabel("Validation macro-F1 (10 split seeds)")
plt.show()

**Answer:** Softmax regression scores **0.667 ± 0.020** across the 10 splits, and its worst split is still far above both naive benchmarks and the pre-registered bar of 0.272. The official seed-42 score (0.661) sits in the middle of that spread. A difference of about 0.02 between two models on a single split is within this noise, so in Milestones 4–5 we'll compare models over the same seeds, not on one split.

## Part 8 - Start / Stop / Continue
- **Start:**
  - Fit kNN and a decision tree (Labs 3–4) on this exact split and seed set for Milestone 4, with the same macro-F1 protocol.
  - Treat the number of GD iterations as a hyperparameter, or use a small $\lambda > 0$ ($\lambda = 0.001$ loses about 0.015 macro-F1 but has a single reproducible optimum), since our $\lambda = 0$ result depends on early stopping.
  - Try the imbalance fixes for Enrolled (recall 0.35): class weights vs Lab 2 undersampling, compared on validation.
  - Run the subgroup analysis planned in M2 §7.6.
- **Stop:**
  - Reporting accuracy as the headline number. The majority benchmark's 50% accuracy shows why.
  - Keeping preprocessing only in notebook cells. Everything now goes through `src/`, so M4 models get exactly the same inputs.
- **Continue:**
  - Selecting on validation only, with the test set untouched until M5.
  - The fixed seed, the dataset hash, the pinned environment, the appended `results/experiment_log.csv`, and the `tests/` checks on every change.

## AI Use Disclosure
Claude Code (Anthropic) helped draft the `src/` modules, the tests, this notebook and the memo. Every number above was produced by running the code in this repository. The code is checked by `tests/` (including a finite-difference gradient check) and by an independent scikit-learn fit (Part 6). See `AI_USE.md`.